# Sub-finding — B2B UPI: The Revolution That Wasn't
**Study:** Mapping UPI: Scale, Geography, and Competition in India's Payment Ecosystem
**Data:** NPCI App-wise Statistics (2020–2026), RBI Payment System Indicators (2019–2026)
**Analysis:** B2B UPI value vs NEFT+RTGS legacy rails — testing whether UPI disrupted business-to-business payments

In [ ]:
import subprocess, sys

for pkg in ["matplotlib", "seaborn", "pandas", "numpy", "sqlalchemy", "pymysql"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from sqlalchemy import create_engine
from pathlib import Path

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.6)
plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("All dependencies ready.")

In [ ]:
# MySQL connection — set your own credentials via environment or edit below.
DB_USER = "your_username"    # your MySQL username
DB_PASS = "your_password"    # your MySQL password
DB_HOST = "localhost"        # or your MySQL host
DB_NAME = "upi_data"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4"
)

# B2B UPI monthly aggregated across all apps — source: NPCI App-wise Statistics
b2b = pd.read_sql("""
    SELECT month_year,
           SUM(b2b_volume_mn) AS b2b_vol,
           SUM(b2b_value_cr)  AS b2b_val,
           SUM(total_volume_mn) AS total_vol,
           SUM(total_value_cr)  AS total_val
    FROM upi_apps
    GROUP BY month_year
    ORDER BY month_year
""", engine)

# NEFT + RTGS monthly — source: RBI Payment System Indicators
legacy = pd.read_sql("""
    SELECT month_year, instrument, volume_mn, value_cr
    FROM rbi_instruments
    ORDER BY month_year, instrument
""", engine)

# Clean B2B: drop null/zero b2b_val rows
b2b = b2b.dropna(subset=["b2b_val"]).copy()
b2b = b2b[b2b["b2b_val"] > 0].copy()
b2b["date"] = pd.to_datetime(b2b["month_year"] + "-01")
b2b = b2b.sort_values("month_year").reset_index(drop=True)

legacy["date"] = pd.to_datetime(legacy["month_year"] + "-01")

print(f"B2B data   : {len(b2b)} months  ({b2b['month_year'].min()} -> {b2b['month_year'].max()})")
print(f"Legacy data: {len(legacy)} rows  ({legacy['month_year'].min()} -> {legacy['month_year'].max()})")
print(f"Instruments: {sorted(legacy['instrument'].unique())}")
print("\nLatest B2B rows:")
print(b2b.tail(4).to_string(index=False))

In [ ]:
# 1. B2B share of total UPI value each month
b2b["b2b_share_pct"] = b2b["b2b_val"] / b2b["total_val"] * 100

# 2. Combined NEFT+RTGS value per month
leg_pivot = (
    legacy.pivot_table(
        index=["month_year", "date"],
        columns="instrument",
        values="value_cr",
        aggfunc="sum"
    ).reset_index()
)
leg_pivot.columns.name = None
for col in ["NEFT", "RTGS"]:
    if col not in leg_pivot.columns:
        leg_pivot[col] = np.nan
leg_pivot["neft_rtgs_val"] = leg_pivot[["NEFT", "RTGS"]].sum(axis=1, min_count=1)
leg_pivot = leg_pivot.sort_values("month_year").reset_index(drop=True)

# Merge B2B and legacy on month_year (inner = only months with both datasets)
merged = b2b.merge(
    leg_pivot[["month_year", "NEFT", "RTGS", "neft_rtgs_val"]],
    on="month_year", how="inner"
)
merged["date"] = pd.to_datetime(merged["month_year"] + "-01")
merged = merged.sort_values("month_year").reset_index(drop=True)

# 3 & 4. YoY growth rates
merged["b2b_yoy"]       = merged["b2b_val"].pct_change(12) * 100
merged["neft_rtgs_yoy"] = merged["neft_rtgs_val"].pct_change(12) * 100

leg_pivot["neft_yoy"] = leg_pivot["NEFT"].pct_change(12) * 100
leg_pivot["rtgs_yoy"] = leg_pivot["RTGS"].pct_change(12) * 100

growth = merged[["month_year", "date", "b2b_yoy"]].merge(
    leg_pivot[["month_year", "neft_yoy", "rtgs_yoy"]], on="month_year", how="left"
)

# 5. Crossover detection
b2b_vs_rtgs     = merged[merged["b2b_val"] > merged["RTGS"]]
b2b_vs_combined = merged[merged["b2b_val"] > merged["neft_rtgs_val"]]

b2b_rtgs_cross     = b2b_vs_rtgs.iloc[0]["month_year"]     if not b2b_vs_rtgs.empty     else None
b2b_combined_cross = b2b_vs_combined.iloc[0]["month_year"] if not b2b_vs_combined.empty else None

print(f"Merged months  : {len(merged)}  ({merged['month_year'].min()} -> {merged['month_year'].max()})")
print(f"B2B > RTGS alone  : {b2b_rtgs_cross or 'Not yet'}")
print(f"B2B > NEFT+RTGS   : {b2b_combined_cross or 'Not yet'}")
print(f"Latest B2B share  : {merged['b2b_share_pct'].iloc[-1]:.1f}% of total UPI value")

## Chart 1 — B2B UPI vs NEFT + RTGS (Value in ₹ Crore)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Shaded fills
ax.fill_between(merged["date"], merged["neft_rtgs_val"],
                alpha=0.07, color="#374151")
ax.fill_between(merged["date"], merged["b2b_val"],
                alpha=0.12, color="#5b21b6")

# Main lines
ax.plot(merged["date"], merged["neft_rtgs_val"],
        color="#374151", lw=3.0, label="NEFT + RTGS (combined)")
ax.plot(merged["date"], merged["b2b_val"],
        color="#5b21b6", lw=3.0, label="B2B UPI")

# RTGS dashed reference
ax.plot(merged["date"], merged["RTGS"],
        color="#64748b", lw=2.0, ls="--", alpha=0.65, label="RTGS only")

# Mark B2B vs NEFT+RTGS crossover
if b2b_combined_cross:
    cx = merged[merged["month_year"] == b2b_combined_cross].iloc[0]
    ax.axvline(cx["date"], color="#dc2626", lw=1.5, ls="--", alpha=0.65)
    ax.annotate(
        f"B2B UPI exceeds\nNEFT+RTGS\n{b2b_combined_cross}",
        xy=(cx["date"], cx["b2b_val"]),
        xytext=(22, 34), textcoords="offset points",
        fontsize=13, color="#dc2626",
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="#dc2626", alpha=0.92),
        arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.3)
    )

# Mark B2B vs RTGS-alone crossover
if b2b_rtgs_cross:
    rx = merged[merged["month_year"] == b2b_rtgs_cross].iloc[0]
    ax.scatter([rx["date"]], [rx["b2b_val"]],
               color="#7c3aed", s=160, zorder=6,
               edgecolors="white", linewidths=2.0)
    ax.annotate(
        f"B2B UPI exceeds\nRTGS alone\n{b2b_rtgs_cross}",
        xy=(rx["date"], rx["b2b_val"]),
        xytext=(-140, -60), textcoords="offset points",
        fontsize=13, color="#7c3aed",
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="#7c3aed", alpha=0.92),
        arrowprops=dict(arrowstyle="->", color="#7c3aed", lw=1.3)
    )

# End-of-line labels
last = merged.iloc[-1]
ax.annotate(
    f"NEFT+RTGS:\n₹{last['neft_rtgs_val']:,.0f} Cr",
    xy=(last["date"], last["neft_rtgs_val"]),
    xytext=(-155, 16), textcoords="offset points",
    fontsize=13, color="#374151", fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#374151", alpha=0.92),
    arrowprops=dict(arrowstyle="->", color="#374151", lw=1.2)
)
ax.annotate(
    f"B2B UPI:\n₹{last['b2b_val']:,.0f} Cr",
    xy=(last["date"], last["b2b_val"]),
    xytext=(-140, -55), textcoords="offset points",
    fontsize=13, color="#5b21b6", fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#5b21b6", alpha=0.92),
    arrowprops=dict(arrowstyle="->", color="#5b21b6", lw=1.2)
)

ax.set_xlim(merged["date"].min(), merged["date"].max())
ax.set_ylim(0)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"₹{y/1e5:.1f}L Cr" if y >= 1e5 else f"₹{y:,.0f} Cr"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("Value (₹ Crore)", fontsize=16)
ax.set_title("B2B UPI vs Legacy Payment Rails — Monthly Transaction Value",
             fontsize=20, fontweight="bold", pad=16)
ax.legend(fontsize=13, framealpha=0.95, loc="upper left", edgecolor="#e5e7eb")
ax.grid(axis="y", alpha=0.3, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "b2b_vs_legacy.png")
plt.show()
print("Saved -> output/b2b_vs_legacy.png")

## Chart 2 — B2B UPI as % of Total UPI Value

In [ ]:
fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Color bars by share level: light → dark purple
vmin, vmax = b2b["b2b_share_pct"].min(), b2b["b2b_share_pct"].max()
norm = (b2b["b2b_share_pct"] - vmin) / (vmax - vmin)
bar_colors = plt.cm.Purples(0.3 + norm * 0.6)

ax.bar(b2b["date"], b2b["b2b_share_pct"], width=25,
       color=bar_colors, alpha=0.88)

# Linear trend overlay
x_num  = (b2b["date"] - b2b["date"].min()).dt.days.values
coeffs = np.polyfit(x_num, b2b["b2b_share_pct"].values, deg=1)
trend  = np.poly1d(coeffs)(x_num)
ax.plot(b2b["date"], trend,
        color="#dc2626", lw=2.5, ls="--", alpha=0.80, label="Linear trend")

# Paytm nodal account termination — Mar 2024 (when B2B volumes collapsed)
paytm_date = pd.Timestamp("2024-03-01")
ax.axvline(paytm_date, color="#dc2626", lw=1.8, ls="--", alpha=0.75)
y_top = b2b["b2b_share_pct"].max() * 1.35
ax.annotate(
    "Paytm nodal accounts terminated\n(Feb-Mar 2024) -- B2B volumes collapse",
    xy=(paytm_date, y_top * 0.72),
    xytext=(14, 0), textcoords="offset points",
    fontsize=13, color="#dc2626", fontweight="bold", va="center",
    bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="#dc2626", alpha=0.92),
)

# Latest value callout
ax.annotate(
    f"{b2b['b2b_share_pct'].iloc[-1]:.1f}%",
    xy=(b2b["date"].iloc[-1], b2b["b2b_share_pct"].iloc[-1]),
    xytext=(0, 14), textcoords="offset points",
    ha="center", fontsize=14, fontweight="bold", color="#4c1d95",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#4c1d95", alpha=0.92)
)

# Peak annotation
peak_row = b2b.loc[b2b["b2b_share_pct"].idxmax()]
ax.annotate(
    f"Peak: {peak_row['b2b_share_pct']:.1f}%\n{peak_row['month_year']}",
    xy=(peak_row["date"], peak_row["b2b_share_pct"]),
    xytext=(0, 16), textcoords="offset points",
    ha="center", fontsize=13, color="#7c3aed",
    bbox=dict(boxstyle="round,pad=0.4", fc="#ede9fe", ec="#7c3aed", alpha=0.85)
)

ax.set_xlim(b2b["date"].min() - pd.DateOffset(months=1),
            b2b["date"].max()  + pd.DateOffset(months=2))
ax.set_ylim(0, y_top)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.tick_params(axis="x", rotation=30, labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.set_ylabel("B2B Share of Total UPI Value (%)", fontsize=16)
ax.set_title("B2B UPI — Growing Share of Total UPI Transaction Value",
             fontsize=20, fontweight="bold", pad=16)
ax.legend(fontsize=13, framealpha=0.95, loc="upper left", edgecolor="#e5e7eb")
ax.grid(axis="y", alpha=0.3, lw=0.6)
ax.grid(axis="x", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "b2b_share.png")
plt.show()
print("Saved -> output/b2b_share.png")

## Summary Statistics

In [ ]:
last = merged.iloc[-1]
latest_b2b_share      = last["b2b_share_pct"]
latest_b2b_val        = last["b2b_val"]
latest_neft_rtgs_val  = last["neft_rtgs_val"]
b2b_pct_of_neft_rtgs  = latest_b2b_val / latest_neft_rtgs_val * 100

print("=" * 62)
print("  FINDING 2 — B2B UPI vs LEGACY RAILS")
print("=" * 62)
print(f"  Latest month                       : {merged['month_year'].iloc[-1]}")
print(f"  B2B UPI share of total UPI value   : {latest_b2b_share:.1f}%")
print(f"  B2B UPI value (latest month)       : \u20b9{latest_b2b_val:,.0f} Cr")
print(f"  NEFT + RTGS combined value         : \u20b9{latest_neft_rtgs_val:,.0f} Cr")
print(f"  B2B UPI as % of NEFT+RTGS          : {b2b_pct_of_neft_rtgs:.1f}%")
print()
if b2b_rtgs_cross:
    print(f"  [YES] B2B UPI first exceeded RTGS alone  : {b2b_rtgs_cross}")
else:
    print(f"  [NO]  B2B UPI has not yet exceeded RTGS alone")
if b2b_combined_cross:
    print(f"  [YES] B2B UPI first exceeded NEFT+RTGS   : {b2b_combined_cross}")
else:
    print(f"  [NO]  B2B UPI has not yet exceeded NEFT+RTGS combined")
print("=" * 62)